# Phase 0 spike — does a live vLLM expose a MOVING prefix-cache hit rate?

**This notebook is the gate.** `CHECKPOINTS.md` Phase 0: no simulator number is trustworthy until this
passes. Run it top to bottom on the A100/H100.

**It is self-contained.** It does not import `kv_routing_sim` and needs no repo, no dataset, no HF token.
Only `vllm` + the standard library. Copy this one file to the box and run it.

---

## What it answers, in order of how badly a 'no' would hurt

| # | Question | Why it matters |
|---|---|---|
| 1 | Do `vllm:prefix_cache_queries` and `vllm:prefix_cache_hits` **exist**? | ADR-008. `vllm:prefix_cache_hit_rate` does **not** exist — a competitor's README cites it, which is how you can tell they never ran the engine |
| 2 | Do they **move** when a shared-prefix workload arrives? | Present-but-flat is a silent failure and the one most likely to be mistaken for success |
| 3 | ⭐ Does `queries` count **BLOCKS or TOKENS**? | We count blocks. Comparing a block-rate to a token-rate gives a confident, wrong calibration error. **Nobody has checked this.** |
| 4 | Is the hit rate much higher with caching **on** vs **off**? | Proves the mechanism, not just the plumbing |
| 5 | Do the three `vllm:kv_block_*` histograms appear? | **Hypothesis H4 is gated on this.** They turn 'the residual is eviction skew' from a story into a measurement |
| 6 | Can **two servers coexist** on one GPU? | Phase 6's 2-replica validation. Never attempted |

## Expected runtime ≈ 25–35 min

pip install vllm 3–8 min · model download 1–2 min · each server start 1–3 min · workloads seconds.

## Which model

**Spike: `Qwen/Qwen3-0.6B`** — ungated (no token, no licence click), ~1.5 GB, 28 layers / 8 KV heads /
head_dim 128 → **112 KiB of KV per token**. Small enough to start fast; the spike only needs the counters
to exist and move, not a realistic model.

**Later, for the Phase-3 calibration: `Qwen/Qwen3-8B`** — also ungated, 36 layers → **144 KiB/token**.
The cost coefficients must come from a model shape people actually serve.

⛔ **Do not use `meta-llama/*` here.** It is `gated=manual`: licence acceptance plus a token, which is a
pointless dependency for a spike.


## 0 — Configuration

The only cell you should need to edit.


In [ ]:
MODEL = "Qwen/Qwen3-0.6B"   # ungated. Qwen3-8B for the Phase-3 calibration.
BLOCK_SIZE = 16             # vLLM default; the block granularity our cache model assumes
PORT_A = 8000
PORT_B = 8001               # second server, for the coexistence test
N_SHARED_REQUESTS = 64      # requests sharing one long prefix
SHARED_PREFIX_TOKENS = 512  # = 32 full blocks at BLOCK_SIZE 16
GPU_MEM_UTIL_SINGLE = 0.85
GPU_MEM_UTIL_SPLIT = 0.42   # each, when testing two servers on one un-partitioned card
STARTUP_TIMEOUT_S = 900
OUT_DIR = "phase0_results"  # everything lands here; copy it back to artifacts/results/


## 1 — What are we actually on?

Recorded into every results file. The GPU and driver are README content — a calibration report that
does not name its hardware reads as never having run.


In [ ]:
import json, os, pathlib, platform, re, subprocess, sys, time, urllib.error, urllib.request
from datetime import datetime, timezone

pathlib.Path(OUT_DIR).mkdir(exist_ok=True)

def sh(cmd, timeout=60):
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        return r.stdout.strip()
    except Exception as e:
        return f"<failed: {e}>"

ENV = {
    "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "platform": platform.platform(),
    "python": sys.version.split()[0],
    "cpu_count": os.cpu_count(),
    "nvidia_smi_gpus": sh("nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv,noheader"),
    "nvidia_smi_L": sh("nvidia-smi -L"),
    "mig_mode": sh("nvidia-smi --query-gpu=mig.mode.current --format=csv,noheader"),
    "cuda_version": sh("nvcc --version | tail -1"),
}
for k, v in ENV.items():
    print(f"{k:22s} {v}")

MIG_UUIDS = [ln.split("UUID:")[1].strip().rstrip(")").strip()
             for ln in ENV["nvidia_smi_L"].splitlines()
             if "MIG-" in ln and "UUID:" in ln]
print(f"\nMIG instances found: {len(MIG_UUIDS)}")
if len(MIG_UUIDS) < 2:
    print("  -> fewer than 2, so the coexistence test will use the MEMORY-SPLIT fallback.")
    print("  -> ADR-005: hit rate is reportable from that path, TTFT percentiles are NOT.")


## 2 — Install vLLM

⚠️ Slow (3–8 min). Skip if the image already has it. The **version that actually installs** is the
thing that matters, not the version we planned against.


In [ ]:
# Comment out if vllm is already present.
!pip install -q vllm


## 3 — ⛔ Audit the metric names against THIS version's own source

The most important cell in the notebook, and it takes two seconds.

We planned against vLLM v0.29.0. If the box installed something else, the metric names must be re-read
from *its* `v1/metrics/` before any number is trusted. This is exactly the step
`Mike-Ew/kv-cache-routing` skipped: their README cites `vllm:prefix_cache_hit_rate`, a V0-era gauge that
no longer exists, which is the tell that they planned against docs and never started a server.


In [ ]:
import vllm
VLLM_VERSION = vllm.__version__
print("vllm version:", VLLM_VERSION)

metrics_dir = pathlib.Path(vllm.__file__).parent / "v1" / "metrics"
print("metrics source:", metrics_dir, "exists:", metrics_dir.exists())

blob = ""
if metrics_dir.exists():
    for p in metrics_dir.rglob("*.py"):
        blob += p.read_text(errors="replace")

WANTED = ["prefix_cache_queries", "prefix_cache_hits", "time_to_first_token_seconds",
          "num_requests_waiting", "request_prefill_kv_computed_tokens",
          "kv_block_lifetime_seconds", "kv_block_idle_before_evict_seconds",
          "kv_block_reuse_gap_seconds"]
AUDIT = {name: (name in blob) for name in WANTED}
# Expected ABSENT. If this is ever True, ADR-008 is wrong on this version and must be amended.
AUDIT["prefix_cache_hit_rate_EXPECTED_ABSENT"] = "prefix_cache_hit_rate" in blob

for k, v in AUDIT.items():
    mark = "OK " if (v and "EXPECTED_ABSENT" not in k) else ("!! " if v else "-- ")
    print(f"  {mark} {k}: {v}")

if AUDIT["prefix_cache_hit_rate_EXPECTED_ABSENT"]:
    print("\n!! prefix_cache_hit_rate APPEARS in this version. ADR-008 needs a dated amendment.")
if not (AUDIT["prefix_cache_queries"] and AUDIT["prefix_cache_hits"]):
    print("\n!! The two counters we depend on are NOT in this version. STOP and re-derive the audit.")


## 4 — Does this build have `--kv-cache-metrics-sample`?

**H4 is gated on this flag.** It exposes the three `kv_block_*` histograms that turn the calibration
residual from a narrative into a measurement. Probed *before* spending a server start on it.


In [ ]:
help_txt = sh("vllm serve --help", timeout=300)
KV_SAMPLE_FLAG = "--kv-cache-metrics-sample" in help_txt
print("--kv-cache-metrics-sample supported:", KV_SAMPLE_FLAG)
if not KV_SAMPLE_FLAG:
    print("  -> the three vllm:kv_block_* histograms are unavailable on this build.")
    print("  -> HYPOTHESIS H4 IS UNTESTABLE AS WRITTEN. Amend docs/09-hypotheses.md with a date;")
    print("     do NOT keep the narrative.")


## 5 — Server helpers

A vLLM server is launched with `subprocess.Popen`, which survives across notebook cells as long as the
kernel lives. Logs are tee'd to a file so a failed start can be diagnosed instead of guessed at.


In [ ]:
SERVERS = {}   # port -> dict(proc, log_path, argv)

def start_server(port, prefix_caching=True, device=None, gpu_mem_util=GPU_MEM_UTIL_SINGLE, extra=None):
    argv = ["vllm", "serve", MODEL, "--port", str(port),
            "--block-size", str(BLOCK_SIZE),
            "--gpu-memory-utilization", str(gpu_mem_util)]
    # V1 has prefix caching ON by default, so the OFF arm needs the explicit negation.
    argv += ["--enable-prefix-caching"] if prefix_caching else ["--no-enable-prefix-caching"]
    if prefix_caching and KV_SAMPLE_FLAG:
        argv += ["--kv-cache-metrics-sample", "1.0"]
    if extra:
        argv += extra

    env = dict(os.environ)
    if device is not None:
        env["CUDA_VISIBLE_DEVICES"] = device

    log_path = pathlib.Path(OUT_DIR) / f"server_{port}.log"
    log = open(log_path, "w")
    proc = subprocess.Popen(argv, stdout=log, stderr=subprocess.STDOUT, env=env)
    SERVERS[port] = {"proc": proc, "log": log, "log_path": str(log_path), "argv": argv}
    print("launched:", " ".join(argv), f"(CUDA_VISIBLE_DEVICES={device})" if device else "")
    return proc

def http_get(url, timeout=10):
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r:
            return r.status, r.read().decode("utf-8", errors="replace")
    except urllib.error.HTTPError as e:
        return e.code, ""
    except Exception as e:
        return 0, str(e)

def http_post(url, payload, timeout=300):
    data = json.dumps(payload).encode()
    req = urllib.request.Request(url, data=data, headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return r.status, r.read().decode("utf-8", errors="replace")
    except urllib.error.HTTPError as e:
        return e.code, e.read().decode("utf-8", errors="replace")
    except Exception as e:
        return 0, str(e)

def wait_ready(port, timeout=STARTUP_TIMEOUT_S, progress_every=30):
    """Poll /health until the server answers. Heartbeat while waiting; log tail on failure.

    The heartbeat matters: a first run downloads weights and compiles, so a quiet cell for two
    minutes is normal and a quiet cell for ten is a problem. Without it you cannot tell which
    one you are in — while the log that explains it is already on disk.
    """
    t0 = time.time()
    _last = [0.0]
    while time.time() - t0 < timeout:
        if http_get(f"http://127.0.0.1:{port}/health", 5)[0] == 200:
            print(f"  port {port} ready in {time.time()-t0:.0f}s")
            return True
        waited = time.time() - t0
        if waited - _last[0] >= progress_every:
            _last[0] = waited
            tail = pathlib.Path(SERVERS[port]["log_path"]).read_text(errors="replace").strip()
            hint = tail.splitlines()[-1][:110] if tail else "(log still empty)"
            print(f"    ...{waited:.0f}s  {hint}")
        if SERVERS.get(port) and SERVERS[port]["proc"].poll() is not None:
            rc = SERVERS[port]["proc"].returncode
            print(f"  !! server on {port} EXITED (returncode {rc}). Log tail:")
            print(pathlib.Path(SERVERS[port]["log_path"]).read_text(errors="replace")[-2500:])
            return False
        time.sleep(5)
    print(f"  !! port {port} not ready within {timeout}s. Log tail:")
    print(pathlib.Path(SERVERS[port]["log_path"]).read_text(errors="replace")[-2500:])
    return False

def stop_server(port):
    s = SERVERS.pop(port, None)
    if not s:
        return
    s["proc"].terminate()
    try:
        s["proc"].wait(timeout=90)
    except subprocess.TimeoutExpired:
        s["proc"].kill()
    s["log"].close()
    print(f"stopped server on {port}")

def scrape(port):
    return http_get(f"http://127.0.0.1:{port}/metrics", 30)[1]

def metric_sum(text, family):
    """Sum every sample of a metric family, tolerating an optional _total and any labels.

    prometheus_client appends _total to counters, so the scraped line is very likely
    vllm:prefix_cache_queries_total{...}. Hard-coding either spelling returns a confident
    false negative. None means ABSENT, which is different from 0.0.
    """
    pat = re.compile(r"^" + re.escape(family) + r"(?:_total)?(?:\{[^}]*\})?\s+([0-9eE+.\-]+)\s*$", re.M)
    vals = [float(m.group(1)) for m in pat.finditer(text)]
    return sum(vals) if vals else None

def family_present(text, family):
    return bool(re.search(r"^" + re.escape(family), text, re.M))

print("helpers defined")


## 6 — Start the server, prefix caching ON


In [ ]:
start_server(PORT_A, prefix_caching=True)
assert wait_ready(PORT_A), "server did not come up - read the log tail above before continuing"


## 7 — ⭐ BLOCKS or TOKENS? The question nobody has answered

Our cache model counts `queries` and `hits` in **blocks**. If vLLM counts **tokens**, every
predicted-vs-measured comparison is off by a factor of `BLOCK_SIZE` — a confidently wrong calibration
error, which is the worst possible failure for the deliverable this project calls its credential.

**The test:** on a cold cache, send one request whose prompt is an exact number of tokens (sent as token
**ids**, so no tokenizer is involved and the count is not an estimate) and see how much `queries`
increments. 512 tokens = 32 blocks, so the two hypotheses are 16× apart and cannot be confused.


In [ ]:
import random
rng = random.Random(0)

# Token ids well inside any modern vocab. Sent as ids => exact token count, no tokenizer needed.
probe_ids = [rng.randrange(1000, 50000) for _ in range(SHARED_PREFIX_TOKENS)]
expected_blocks = SHARED_PREFIX_TOKENS // BLOCK_SIZE

before = scrape(PORT_A)
q0 = metric_sum(before, "vllm:prefix_cache_queries")
h0 = metric_sum(before, "vllm:prefix_cache_hits")
print(f"before: queries={q0} hits={h0}")
assert q0 is not None, "vllm:prefix_cache_queries is ABSENT - stop, and re-read cell 3"

status, body = http_post(f"http://127.0.0.1:{PORT_A}/v1/completions",
                         {"model": MODEL, "prompt": probe_ids, "max_tokens": 4, "temperature": 0.0})
print("probe request status:", status)
if status != 200:
    print("  body:", body[:400])
    print("  -> if token-id prompts were rejected, fall back to a text prompt and tokenize it")
    print("     with transformers to learn the exact token count.")

time.sleep(2)
after = scrape(PORT_A)
q1 = metric_sum(after, "vllm:prefix_cache_queries")
dq = (q1 or 0) - (q0 or 0)

print(f"\nprompt tokens sent : {SHARED_PREFIX_TOKENS}")
print(f"= full blocks      : {expected_blocks} (at block_size {BLOCK_SIZE})")
print(f"queries increment  : {dq}")

if abs(dq - expected_blocks) <= 2:
    QUERY_UNIT = "blocks"
elif abs(dq - SHARED_PREFIX_TOKENS) <= BLOCK_SIZE:
    QUERY_UNIT = "tokens"
else:
    QUERY_UNIT = f"unclear (increment {dq})"
print(f"\n==> vllm:prefix_cache_queries counts: {QUERY_UNIT.upper()}")
if QUERY_UNIT == "tokens":
    print("    !! our cache model counts BLOCKS. Divide vLLM by block_size before comparing,")
    print("       and record this in ADR-008 with a date.")
elif QUERY_UNIT == "blocks":
    print("    OK - matches cache/block_trie.py. Record the confirmation in ADR-008.")


## 8 — The shared-prefix workload

Every request carries the identical 512-token prefix and a short unique tail — the shape agentic traffic
actually has. After the first, each should hit cache on all 32 shared blocks. If hits do not move here,
they will not move anywhere.


In [ ]:
base = scrape(PORT_A)
qB, hB = metric_sum(base, "vllm:prefix_cache_queries"), metric_sum(base, "vllm:prefix_cache_hits")

t0 = time.time()
errors = []
for i in range(N_SHARED_REQUESTS):
    prompt = probe_ids + [60000 + i]          # identical prefix, unique 1-token tail
    st, bd = http_post(f"http://127.0.0.1:{PORT_A}/v1/completions",
                       {"model": MODEL, "prompt": prompt, "max_tokens": 8, "temperature": 0.0})
    if st != 200:
        errors.append(f"{i}: {st} {bd[:120]}")
wall = time.time() - t0
print(f"sent {N_SHARED_REQUESTS} requests in {wall:.1f}s; errors: {len(errors)}")
for e in errors[:3]:
    print("  ", e)

time.sleep(2)
warm = scrape(PORT_A)
qW, hW = metric_sum(warm, "vllm:prefix_cache_queries"), metric_sum(warm, "vllm:prefix_cache_hits")

dq, dh = (qW or 0) - (qB or 0), (hW or 0) - (hB or 0)
print(f"\nqueries +{dq}   hits +{dh}")
if dq:
    print(f"hit rate over this workload = {dh}/{dq} = {dh / dq:.4f}")
else:
    print("no queries recorded")

MOVED = dq > 0 and dh > 0
print(f"\n==> counters MOVED: {MOVED}")
if not MOVED:
    print("    !! present but flat. That is the silent failure mode. Do NOT proceed to Phase 1.")

HIST = {f: family_present(warm, f) for f in
        ["vllm:kv_block_lifetime_seconds", "vllm:kv_block_idle_before_evict_seconds",
         "vllm:kv_block_reuse_gap_seconds"]}
print("\nkv_block_* histograms (H4 depends on these):")
for k, v in HIST.items():
    print(f"  {'OK' if v else '--'}  {k}: {v}")

pathlib.Path(OUT_DIR, "metrics_caching_on.txt").write_text(warm)
print("\nraw scrape saved (commit this, not just the JSON)")


## 9 — Caching OFF, for comparison

The only arm that proves the *mechanism* rather than the plumbing. Restarts the server with
`--no-enable-prefix-caching` — in V1 prefix caching is ON by default, so omitting the flag would **not**
turn it off and the comparison would be meaningless while still producing two plausible numbers.


In [ ]:
stop_server(PORT_A)
time.sleep(5)
start_server(PORT_A, prefix_caching=False)
ok_off = wait_ready(PORT_A)

RATE_OFF = None
if ok_off:
    b = scrape(PORT_A)
    q0o, h0o = metric_sum(b, "vllm:prefix_cache_queries"), metric_sum(b, "vllm:prefix_cache_hits")
    for i in range(N_SHARED_REQUESTS):
        http_post(f"http://127.0.0.1:{PORT_A}/v1/completions",
                  {"model": MODEL, "prompt": probe_ids + [60000 + i], "max_tokens": 8, "temperature": 0.0})
    time.sleep(2)
    a = scrape(PORT_A)
    q1o, h1o = metric_sum(a, "vllm:prefix_cache_queries"), metric_sum(a, "vllm:prefix_cache_hits")
    dqo, dho = (q1o or 0) - (q0o or 0), (h1o or 0) - (h0o or 0)
    RATE_OFF = (dho / dqo) if dqo else None
    print(f"caching OFF: queries +{dqo} hits +{dho} rate={RATE_OFF}")
    pathlib.Path(OUT_DIR, "metrics_caching_off.txt").write_text(a)

RATE_ON = (dh / dq) if dq else None
print(f"\nON  hit rate: {RATE_ON}")
print(f"OFF hit rate: {RATE_OFF}")
if RATE_ON and RATE_OFF is not None and RATE_ON > RATE_OFF:
    print("==> the mechanism works: caching on is materially higher.")
elif RATE_ON and RATE_OFF is not None:
    print("==> !! caching ON is not beating OFF. Investigate before trusting any sweep.")


## 10 — Can two servers coexist on one GPU?

Phase 6's 2-replica validation depends on this, and it has never been attempted.

⚠️ **ADR-005, and this decides what you may publish:** MIG partitions SMs *and* memory; the memory-split
fallback partitions only memory. So **hit rate is reportable from either path, but TTFT percentiles only
from true MIG isolation** — SM contention in the fallback is an artifact of the test rig, not an effect
of the routing policy. The path is recorded in the results file for exactly this reason.


In [ ]:
stop_server(PORT_A)
time.sleep(5)

if len(MIG_UUIDS) >= 2:
    HARDWARE_PATH = "mig"
    start_server(PORT_A, device=MIG_UUIDS[0], gpu_mem_util=GPU_MEM_UTIL_SINGLE)
    start_server(PORT_B, device=MIG_UUIDS[1], gpu_mem_util=GPU_MEM_UTIL_SINGLE)
else:
    HARDWARE_PATH = "memory-split"
    start_server(PORT_A, gpu_mem_util=GPU_MEM_UTIL_SPLIT)
    start_server(PORT_B, gpu_mem_util=GPU_MEM_UTIL_SPLIT)

a_ok = wait_ready(PORT_A)
b_ok = wait_ready(PORT_B)
# Re-check A: bringing B up can OOM A out from under us, and that only shows on a re-check.
a_still = http_get(f"http://127.0.0.1:{PORT_A}/health", 10)[0] == 200 if a_ok else False
COEXIST = bool(a_ok and b_ok and a_still)

print(f"\npath: {HARDWARE_PATH}")
print(f"A ready: {a_ok}   B ready: {b_ok}   A still healthy after B: {a_still}")
print(f"==> two servers coexist: {COEXIST}")
if COEXIST and HARDWARE_PATH == "memory-split":
    print("    hit rate is reportable from this path. TTFT percentiles are NOT (ADR-005).")
if not COEXIST:
    print("    Phase 6 takes its documented fallback: single-replica calibration plus a")
    print("    reproducible account of why 2 replicas failed. Log the cause in the upstream log.")


## 11 — Save the verdict

Copy `phase0_results/` back into the repo's `artifacts/results/` and commit it — **including the raw
`.txt` scrapes**, which are the evidence. The JSON alone is a parse; the scrape is the thing parsed.


In [ ]:
GATE = {
    "counters_present": bool(AUDIT["prefix_cache_queries"] and AUDIT["prefix_cache_hits"]),
    "counters_moved": bool(MOVED),
    "hit_rate_metric_correctly_absent": not AUDIT["prefix_cache_hit_rate_EXPECTED_ABSENT"],
    "query_unit": QUERY_UNIT,
    "caching_on_beats_off": bool(RATE_ON and RATE_OFF is not None and RATE_ON > RATE_OFF),
    "kv_block_histograms_present": HIST,
    "kv_sample_flag_supported": KV_SAMPLE_FLAG,
    "two_servers_coexist": COEXIST,
    "hardware_path": HARDWARE_PATH,
}
GATE["PHASE_0_PASSED"] = GATE["counters_present"] and GATE["counters_moved"]

payload = {
    "script": "notebooks/phase0_spike_vllm.ipynb",
    "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "model": MODEL,
    "vllm_version": VLLM_VERSION,
    "block_size": BLOCK_SIZE,
    "environment": ENV,
    "metric_name_audit": AUDIT,
    "hit_rate_on": RATE_ON,
    "hit_rate_off": RATE_OFF,
    "gate": GATE,
    "server_argv": {str(p): s["argv"] for p, s in SERVERS.items()},
}
out = pathlib.Path(OUT_DIR, "phase0_spike.json")
out.write_text(json.dumps(payload, indent=2))

print(json.dumps(GATE, indent=2))
print(f"\nwrote {out}")
print("\nfiles to copy back into artifacts/results/:")
for f in sorted(pathlib.Path(OUT_DIR).iterdir()):
    print(f"  {f.name}  ({f.stat().st_size:,} B)")

print("\n" + ("=" * 72))
if GATE["PHASE_0_PASSED"]:
    print("PHASE 0 GATE PASSED. Simulator code is unblocked.")
    print(f"  - queries counts {QUERY_UNIT} -> record in ADR-008")
    print(f"  - H4 testable: {all(HIST.values())}")
    print(f"  - 2 replicas: {COEXIST} via {HARDWARE_PATH}")
else:
    print("PHASE 0 GATE FAILED. Do not trust any simulator number yet.")
    print("  Log the cause in the upstream log - it may be worth reporting upstream.")
print("=" * 72)


## 12 — Clean up

⚠️ **Run this.** On rented hardware a forgotten vLLM process bills until the box dies.


In [ ]:
for port in list(SERVERS):
    stop_server(port)
print(sh("nvidia-smi --query-compute-apps=pid,used_memory --format=csv"))
print("\nIf any vllm process survives:  !pkill -f 'vllm serve'")
